# Лабораторна робота № 3
## Діагностика шкал вимірювання

Курс СК26 «Інтелектуальний аналіз та візуалізація даних»

**Тривалість:** 2 год аудиторно + 3 год самостійно · **Оцінка:** 4 бали
**Файли:** `Ковід_дані_навчальні.xlsx`, `ЛР3_помилковий_аналіз.ipynb`

---

Шкала вимірювання визначає, які операції над змінною взагалі мають сенс.
Комп'ютер цього не знає: він порахує середнє від номера телефону й не поскаржиться.
Знати має вимірювач.

У цій роботі ви складаєте **паспорт восьми змінних** свого варіанта, показуєте на
власних даних, які статистики для них недопустимі, і виправляєте чужий аналіз,
де таких помилок десять.

In [1]:
ПРІЗВИЩЕ_ІМЯ = 'Ільящук Марта'      # напишіть тут своє прізвище та ім'я
ГРУПА        = 'ШІ-31'
НОМЕР        = 7       # ваш номер за списком групи (1–30)

assert ПРІЗВИЩЕ_ІМЯ and ГРУПА and НОМЕР, 'Заповніть три поля вище'
assert 1 <= НОМЕР <= 30, 'Номер має бути від 1 до 30'

In [2]:
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_colwidth', 60)

ВАРІАНТИ = json.load(open('lr3_variants_student.json', encoding='utf-8'))
В = ВАРІАНТИ[НОМЕР - 1]

print('=' * 74)
print(f'ВАРІАНТ {В["варіант"]}   ·   {ПРІЗВИЩЕ_ІМЯ}   ·   {ГРУПА}')
print('=' * 74)
print()
print('Ваші вісім стовпців:')
for i, c in enumerate(В['стовпці'], 1):
    print(f'   {i}. {c}')
print()
print('Ваша пара для аналізу зв’язку:', ' та '.join(В['пара']))
print()
print('Ваше додаткове питання:')
print('  ', В['питання'])

ВАРІАНТ 7   ·   Ільящук Марта   ·   ШІ-31

Ваші вісім стовпців:
   1. ПІП
   2. помер/жив
   3. ступінь КТ
   4. Дата розтину
   5. КТ
   6. Т вище 37,5
   7. Сечов1
   8. БРпр1

Ваша пара для аналізу зв’язку: Стать та помер/жив

Ваше додаткове питання:
   Ви хочете нормалізувати всі числові стовпці формулою (x − min) / (max − min). Знайдіть у файлі стовпець, на якому ця формула зламається, і поясніть чому.


In [3]:
ФАЙЛ = 'Ковід_дані_навчальні.xlsx'
df = pd.read_excel(ФАЙЛ, sheet_name='Лист1', engine='openpyxl')
МОЇ = В['стовпці']

print('Розмір набору:', df.shape)
df[МОЇ].head(8)

Розмір набору: (121, 77)


,ПІП,помер/жив,ступінь КТ,Дата розтину,КТ,"Т вище 37,5",Сечов1,БРпр1
0,Федорюк Петро Юрійович,1,сердній,2022-03-11,20-30%,3,5.1,3.4
1,Загул Григорій Сергійович,2,середній,NaT,30-35%,3,6.6,2.8
2,Бондара Лариса Павлівна,2,критичний,NaT,70-80%,3,7.0,2.8
3,Ватаманюка Уляна Дмитрівна,2,критичний,NaT,0.8,6,9.5,3.4
4,Бабійа Лариса Олегівна,2,NaN,NaT,NaN,5 до гос,5.2,3.4
5,Ковалика ганна Андріївна,1,середній,2021-12-26,30-40%,NaN,14.0,3.4
6,Височан Віктор Володимирович,2,важкий,NaT,60-65%,NaN,5.7,5.1
7,Дзюбаа Ірина Іванівна,2,легкий,NaT,до 17%,NaN,9.1,5.1


---
# Етап 1. Паспорт змінних

Для кожного з восьми стовпців визначте шкалу вимірювання. Спирайтеся не на тип даних
у pandas, а на **зміст величини**: що означають її значення і які порівняння між ними
мають сенс.

Нагадування:

| Шкала | Ознака |
|---|---|
| Номінальна | значення можна лише порівняти на рівність |
| Порядкова | є природний порядок, але відстані між сусідніми рівнями невідомі |
| Інтервальна | різниці мають сенс, нуль умовний |
| Відношень | є природний нуль, відношення мають сенс |
| Абсолютна | природні і нуль, і одиниця вимірювання |

Окремо позначайте **ідентифікатори** — значення, що не є величиною взагалі
(номер історії хвороби, прізвище, телефон).

In [4]:
# Спершу подивіться, що каже pandas — і не довіряйте йому
df[МОЇ].info()

<class 'pandas.DataFrame'>
RangeIndex: 121 entries, 0 to 120
Data columns (total 8 columns):
 #   Column        Non-Null Count  Dtype         
---  ------        --------------  -----         
 0   ПІП           121 non-null    str           
 1   помер/жив     121 non-null    int64         
 2   ступінь КТ    42 non-null     str           
 3   Дата розтину  61 non-null     datetime64[us]
 4   КТ            44 non-null     object        
 5   Т вище 37,5   78 non-null     object        
 6   Сечов1        117 non-null    float64       
 7   БРпр1         109 non-null    float64       
dtypes: datetime64[us](1), float64(2), int64(1), object(2), str(2)
memory usage: 7.7+ KB


In [5]:
# Далі — що є всередині насправді
for c in МОЇ:
    з = df[c].dropna()
    типи = ', '.join(f'{t.__name__}×{n}' for t, n in з.map(type).value_counts().items())
    print(f'--- {c}')
    print(f'    непорожніх {len(з)} з {len(df)} | унікальних {з.nunique()} | типи: {типи}')
    print(f'    приклади: {[repr(v) for v in з.unique()[:4]]}')
    print()

--- ПІП
    непорожніх 121 з 121 | унікальних 121 | типи: str×121
    приклади: ["'Федорюк Петро Юрійович'", "'Загул Григорій Сергійович'", "'Бондара Лариса Павлівна'", "'Ватаманюка Уляна Дмитрівна'"]

--- помер/жив
    непорожніх 121 з 121 | унікальних 2 | типи: int×121
    приклади: ['np.int64(1)', 'np.int64(2)']

--- ступінь КТ
    непорожніх 42 з 121 | унікальних 12 | типи: str×42
    приклади: ["'сердній'", "'середній'", "'критичний'", "'важкий'"]

--- Дата розтину
    непорожніх 61 з 121 | унікальних 50 | типи: Timestamp×61
    приклади: ["Timestamp('2022-03-11 00:00:00')", "Timestamp('2021-12-26 00:00:00')", "Timestamp('2021-11-19 00:00:00')", "Timestamp('2022-05-13 00:00:00')"]

--- КТ
    непорожніх 44 з 121 | унікальних 26 | типи: float×27, str×17
    приклади: ["'20-30%'", "'30-35%'", "'70-80%'", '0.8']

--- Т вище 37,5
    непорожніх 78 з 121 | унікальних 35 | типи: str×48, int×30
    приклади: ['3', '6', "'5 до гос'", "'3 дні в стац'"]

--- Сечов1
    непорожніх 117 з 121 

### Заповніть паспорт

Для кожного стовпця вкажіть шкалу та **обґрунтування** — одне речення про те,
чому саме ця шкала. Обґрунтування оцінюється, назва шкали сама по собі — ні.

In [6]:
ПАСПОРТ = {
    'ПІП': ('ідентифікатор',
            'Прізвище, ім’я та по батькові лише розрізняють пацієнтів: це назва особи, а не величина, тому ні арифметика, ні порядок значень не мають змісту.'),
    'помер/жив': ('номінальна',
            'Дві взаємовиключні категорії (1 = помер, 2 = вижив) без природного порядку; цифри 1 і 2 — лише мітки, їх можна замінити на 0 і 1 чи на слова.'),
    'ступінь КТ': ('порядкова',
            'Ступені легкий < середній < важкий < вкрай важкий < критичний мають природний порядок, але відстані між сусідніми ступенями невідомі й не обов’язково рівні.'),
    'Дата розтину': ('інтервальна',
            'Різниці між датами (у днях) мають сенс, але нуль календаря умовний: немає «нульової дати», тому відношення дат («удвічі пізніше») безглузді.'),
    'КТ': ('відношень',
            'Частка об’єму легень з ураженням: є природний нуль (ураження немає), 40% справді вдвічі більше за 20%; у файлі частки змішано з текстовими діапазонами.'),
    'Т вище 37,5': ('відношень',
            'Кількість днів із температурою вище 37,5 °C: природний нуль (жодного дня), 6 днів — удвічі більше, ніж 3; у файлі числа змішано з текстом на кшталт «3 дні в стац».'),
    'Сечов1': ('відношень',
            'Концентрація сечовини в крові (ммоль/л): нуль означає відсутність речовини, а відношення («удвічі вища концентрація») мають фізичний зміст.'),
    'БРпр1': ('відношень',
            'Концентрація прямого білірубіну (мкмоль/л): природний нуль — відсутність речовини, відношення концентрацій мають зміст; значення кратні 1,7 — це крок лабораторного округлення.'),
}

for c in МОЇ:
    ПАСПОРТ.setdefault(c, ('', ''))

паспорт_df = pd.DataFrame(
    [(c, ПАСПОРТ[c][0], ПАСПОРТ[c][1]) for c in МОЇ],
    columns=['Стовпець', 'Шкала', 'Обґрунтування'])
паспорт_df

,Стовпець,Шкала,Обґрунтування
0,ПІП,ідентифікатор,"Прізвище, ім’я та по батькові лише розрізняють пацієнтів..."
1,помер/жив,номінальна,"Дві взаємовиключні категорії (1 = помер, 2 = вижив) без ..."
2,ступінь КТ,порядкова,Ступені легкий < середній < важкий < вкрай важкий < крит...
3,Дата розтину,інтервальна,"Різниці між датами (у днях) мають сенс, але нуль календа..."
4,КТ,відношень,Частка об’єму легень з ураженням: є природний нуль (ураж...
5,"Т вище 37,5",відношень,"Кількість днів із температурою вище 37,5 °C: природний н..."
6,Сечов1,відношень,Концентрація сечовини в крові (ммоль/л): нуль означає ві...
7,БРпр1,відношень,Концентрація прямого білірубіну (мкмоль/л): природний ну...


In [7]:
# --- САМОПЕРЕВІРКА 1 ---
ДОПУСТИМІ = {'номінальна', 'порядкова', 'інтервальна', 'відношень',
             'абсолютна', 'ідентифікатор'}
for c in МОЇ:
    ш, об = ПАСПОРТ[c]
    assert ш.strip().lower() in ДОПУСТИМІ, f'{c}: шкалу «{ш}» не розпізнано'
    assert len(об.strip()) >= 25, f'{c}: обґрунтування закоротке'
print('Самоперевірка 1 пройдена: усі вісім стовпців мають шкалу та обґрунтування.')

Самоперевірка 1 пройдена: усі вісім стовпців мають шкалу та обґрунтування.


---
# Етап 2. Допустимі статистики

Для кожного стовпця обчисліть **тільки те, що дозволяє його шкала**.

* номінальна та ідентифікатор — мода, частоти
* порядкова — додатково медіана та квантилі
* інтервальна — додатково середнє та стандартне відхилення
* відношень і абсолютна — усе, зокрема коефіцієнт варіації

Якщо для стовпця не можна порахувати нічого, крім частот, — так і напишіть.

### Крок 0. Приведення даних до придатного вигляду

Шкала відома, але самі дані «брудні»: у `ступінь КТ` те саме значення записано по-різному
(«сердній», «серед ступ», «легка»…), `КТ` і `Т вище 37,5` змішують числа з текстом
(«20-30%», «3 дні в стац»), у `Стать` — чотири написання двох категорій. Перш ніж рахувати
статистики, очищуємо. Правила очищення — свідомі рішення, їх варто знати:

* **ступінь КТ** — 12 написань зводимо до п’яти категорій і задаємо порядок явно;
* **КТ** — число лишаємо числом; діапазон «20-30%» замінюємо серединою інтервалу (0,25),
  «до 17%» — значенням 0,17, порожній рядок — пропуском;
* **Т вище 37,5** — із тексту беремо кількість днів (якщо їх два — «7 днів до госп + 2 дні в стац» — сума);
  зауваж: частина записів — дні *до* госпіталізації, частина — *у стаціонарі*, це різні величини;
* **Стать** — уніфікуємо до «ч» / «ж».

In [8]:
import re

ПОРЯДОК_КТ = ['легкий', 'середній', 'важкий', 'вкрай важкий', 'критичний']
ЯРЛИКИ = {'легка': 'легкий', 'сердній': 'середній',
          'серед ступ': 'середній', 'серед ступеня': 'середній'}

def очистити_ступінь(x):
    if not isinstance(x, str):
        return np.nan
    x = x.strip()
    return ЯРЛИКИ.get(x, x)

def частка_кт(x):
    """число лишаємо числом; діапазон -> середина; порожній текст -> NaN"""
    if isinstance(x, (int, float)):
        return float(x)
    числа = [float(v.replace(',', '.')) for v in re.findall(r'\d+(?:[.,]\d+)?', str(x))]
    return float(np.mean(числа)) / 100 if числа else np.nan

def дні_температури(x):
    """число лишаємо числом; з тексту беремо суму знайдених чисел"""
    if isinstance(x, (int, float)):
        return float(x)
    числа = [int(v) for v in re.findall(r'\d+', str(x))]
    return float(sum(числа)) if числа else np.nan

Ч = pd.DataFrame(index=df.index)
Ч['ПІП']          = df['ПІП']
Ч['помер/жив']    = df['помер/жив']
Ч['ступінь КТ']   = pd.Categorical(df['ступінь КТ'].map(очистити_ступінь),
                                   categories=ПОРЯДОК_КТ, ordered=True)
Ч['Дата розтину'] = pd.to_datetime(df['Дата розтину'], errors='coerce')
Ч['КТ']           = df['КТ'].map(частка_кт)
Ч['Т вище 37,5']  = df['Т вище 37,5'].map(дні_температури)
Ч['Сечов1']       = pd.to_numeric(df['Сечов1'], errors='coerce')
Ч['БРпр1']        = pd.to_numeric(df['БРпр1'], errors='coerce')
Ч['Стать']        = df['Стать'].astype(str).str.strip().str[0]

# перевірка, що очищення нічого не загубило
assert set(df['ступінь КТ'].dropna().map(очистити_ступінь)) <= set(ПОРЯДОК_КТ)
assert Ч['ступінь КТ'].notna().sum() == df['ступінь КТ'].notna().sum()
assert set(Ч['Стать']) == {'ч', 'ж'}

print('Було типів у КТ  :', df['КТ'].dropna().map(lambda v: type(v).__name__).value_counts().to_dict())
print('Чисел у КТ після очищення  :', Ч['КТ'].notna().sum(), 'з', df['КТ'].notna().sum(), 'непорожніх')
print('Чисел у «Т вище 37,5» після очищення:', Ч['Т вище 37,5'].notna().sum(), 'з', df['Т вище 37,5'].notna().sum())
print('Категорій ступеня КТ:', df['ступінь КТ'].dropna().nunique(), '->', Ч['ступінь КТ'].nunique())
print('Написань статі:', df['Стать'].nunique(), '->', Ч['Стать'].nunique())

Було типів у КТ  : {'float': 27, 'str': 17}
Чисел у КТ після очищення  : 43 з 44 непорожніх
Чисел у «Т вище 37,5» після очищення: 78 з 78
Категорій ступеня КТ: 12 -> 5
Написань статі: 4 -> 2


In [9]:
def допустима_статистика(стовпець, шкала):
    """Повертає словник статистик, дозволених для цієї шкали."""
    з = Ч[стовпець].dropna()
    р = {'непорожніх': len(з), 'унікальних': з.nunique()}
    ш = шкала.strip().lower()

    if ш in ('ідентифікатор',):
        return р

    # номінальна та вище: мода й частоти
    р['мода'] = з.mode().iloc[0]
    р['частоти (топ-5)'] = з.value_counts().head(5).to_dict()
    if ш == 'номінальна':
        return р

    # порядкова: медіана та квантилі (через коди категорій у заданому порядку)
    if ш == 'порядкова':
        кат = з.cat.categories
        коди = з.cat.codes.to_numpy()
        q = lambda p: кат[int(np.quantile(коди, p, method='lower'))]
        р['Q1'], р['медіана'], р['Q3'] = q(.25), q(.5), q(.75)
        return р

    # інтервальна й вище: квантилі, середнє, стандартне відхилення
    р['мінімум'], р['максимум'] = з.min(), з.max()
    р['Q1'], р['медіана'], р['Q3'] = з.quantile(.25), з.median(), з.quantile(.75)
    р['середнє'] = з.mean()
    р['стандартне відхилення'] = з.std()
    if ш == 'інтервальна':
        return р

    # відношень та абсолютна: коефіцієнт варіації
    р['коефіцієнт варіації, %'] = round(100 * з.std() / з.mean(), 1)
    return р


for c in МОЇ:
    print(f'--- {c}  ({ПАСПОРТ[c][0]})')
    for k, v in допустима_статистика(c, ПАСПОРТ[c][0]).items():
        if isinstance(v, float):
            v = round(v, 3)
        print(f'    {k:24}: {v}')
    print()
print('Зауваження: для «ПІП» (ідентифікатор) нічого, крім кількості, рахувати не можна;')
print('для «помер/жив» (1 = помер, 2 = вижив) «середнє» навмисно не рахується.')

--- ПІП  (ідентифікатор)
    непорожніх              : 121
    унікальних              : 121

--- помер/жив  (номінальна)
    непорожніх              : 121
    унікальних              : 2
    мода                    : 1
    частоти (топ-5)         : {1: 61, 2: 60}

--- ступінь КТ  (порядкова)
    непорожніх              : 42
    унікальних              : 5
    мода                    : середній
    частоти (топ-5)         : {'середній': 18, 'важкий': 10, 'легкий': 7, 'вкрай важкий': 4, 'критичний': 3}
    Q1                      : середній
    медіана                 : середній
    Q3                      : важкий

--- Дата розтину  (інтервальна)
    непорожніх              : 61
    унікальних              : 50
    мода                    : 2022-06-07 00:00:00
    частоти (топ-5)         : {Timestamp('2022-06-07 00:00:00'): 3, Timestamp('2022-04-07 00:00:00'): 2, Timestamp('2022-04-27 00:00:00'): 2, Timestamp('2022-04-13 00:00:00'): 2, Timestamp('2022-03-30 00:00:00'): 2}
    мінімум  

---
# Етап 3. Три безглузді числа

`df.describe()` порахує статистики для всього, що схоже на число. Частина цих чисел
не має жодного змісту.

Знайдіть **щонайменше три** такі числа у своєму наборі: обчисліть їх, а потім
поясніть, чому кожне з них безглузде.

In [10]:
df[МОЇ].describe(include='all').T

,count,unique,top,freq,mean,min,25%,50%,75%,max,std
ПІП,121,121,Федорюк Петро Юрійович,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
помер/жив,121.0,NaN,NaN,NaN,1.495868,1.0,1.0,1.0,2.0,2.0,0.502062
ступінь КТ,42,12,середній,10,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Дата розтину,61,NaN,NaN,NaN,2022-04-06 13:22:37.377049,2021-01-06 00:00:00,2022-03-29 00:00:00,2022-04-23 00:00:00,2022-05-07 00:00:00,2022-06-22 00:00:00,NaN
КТ,44.0,26.0,0.3,6.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
"Т вище 37,5",78.0,35.0,3.0,13.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Сечов1,117.0,NaN,NaN,NaN,8.93188,3.1,5.5,7.2,9.5,68.0,8.174368
БРпр1,109.0,NaN,NaN,NaN,4.801468,2.4,3.4,5.1,5.3,10.2,1.575191


In [11]:
# готуємо числа, за якими видно, що саме «безглузде»
кт_сирий = pd.to_numeric(df['КТ'], errors='coerce')          # як це зробив би describe()
т_сирий  = pd.to_numeric(df['Т вище 37,5'], errors='coerce')
коди_кт  = Ч['ступінь КТ'].cat.codes[Ч['ступінь КТ'].notna()]
дати     = Ч['Дата розтину'].dropna()

від_1970 = (дати - pd.Timestamp('1970-01-01')).dt.days
від_2021 = (дати - pd.Timestamp('2021-01-01')).dt.days
cv_1970  = від_1970.std() / від_1970.mean()
cv_2021  = від_2021.std() / від_2021.mean()

БЕЗГЛУЗДІ = [
    ('помер/жив', 'середнє', round(df['помер/жив'].mean(), 3),
     'Номінальна шкала: 1 і 2 — мітки, а не величини. Якщо перекодувати в 0/1, «середнє» стане 0,50, '
     'а проміжного стану між смертю й одужанням узагалі не існує; зміст має лише частка (61 з 121).'),
    ('ступінь КТ', 'середнє кодів 0–4', round(коди_кт.mean(), 2),
     'Порядкова шкала: порядок відомий, відстані між ступенями — ні. Різниця «легкий → середній» '
     'не дорівнює «важкий → вкрай важкий», тож середнє кодів залежить від довільно обраних чисел 0–4.'),
    ('Дата розтину', 'коефіцієнт варіації (відлік від 1970 р.)', round(cv_1970, 5),
     f'Інтервальна шкала: нуль умовний. Те саме розсіювання дат, рахуване від 01.01.2021, дає CV = {cv_2021:.2f}, '
     'тобто «варіація» залежить лише від того, де ми поставили нуль.'),
    ('КТ', 'середнє після to_numeric(errors="coerce")', round(кт_сирий.mean(), 3),
     f'Дані непридатні: стовпець змішує частки й текст; coerce мовчки перетворює {df["КТ"].notna().sum() - кт_сирий.notna().sum()} '
     f'із {df["КТ"].notna().sum()} непорожніх записів на NaN, тож середнє рахується лише за {кт_сирий.notna().sum()} записами й зміщене.'),
    ('Т вище 37,5', 'середнє після to_numeric(errors="coerce")', round(т_сирий.mean(), 2),
     f'Дані непридатні: лише {т_сирий.notna().sum()} із {df["Т вище 37,5"].notna().sum()} записів — чисті числа; решта («3 дні в стац», «5 до гос») '
     'відкинута, а вони описують іншу ситуацію (дні до госпіталізації чи в стаціонарі).'),
]

assert len(БЕЗГЛУЗДІ) >= 3, 'Потрібно щонайменше три приклади'
pd.DataFrame(БЕЗГЛУЗДІ, columns=['Стовпець', 'Статистика', 'Значення', 'Чому безглузде'])

,Стовпець,Статистика,Значення,Чому безглузде
0,помер/жив,середнє,1.4960,"Номінальна шкала: 1 і 2 — мітки, а не величини. Якщо пер..."
1,ступінь КТ,середнє кодів 0–4,1.4800,"Порядкова шкала: порядок відомий, відстані між ступенями..."
2,Дата розтину,коефіцієнт варіації (відлік від 1970 р.),0.0038,Інтервальна шкала: нуль умовний. Те саме розсіювання дат...
3,КТ,"середнє після to_numeric(errors=""coerce"")",0.3960,Дані непридатні: стовпець змішує частки й текст; coerce ...
4,"Т вище 37,5","середнє після to_numeric(errors=""coerce"")",3.4700,Дані непридатні: лише 30 із 78 записів — чисті числа; ре...


---
# Етап 4. Міра зв'язку для вашої пари

Для двох змінних вашого варіанта треба оцінити зв'язок. Міру обирає **пара шкал**,
а не зручність:

| Поєднання шкал | Міра |
|---|---|
| відношень × відношень | кореляція Пірсона |
| порядкова × порядкова | рангова кореляція Спірмена |
| номінальна × номінальна | χ² та коефіцієнт Крамера (V) |
| відношень × бінарна номінальна | точковий бісеріальний коефіцієнт |
| номінальна × порядкова | коефіцієнт Крамера (V) |

Обґрунтуйте вибір, обчисліть міру й поясніть результат. Окремо покажіть, **що
станеться**, якщо застосувати Пірсона там, де він недопустимий, — і чому отримане
число не можна інтерпретувати.

In [12]:
A, B = В['пара']
print('Ваша пара:', A, '×', B)
print()
print(f'{A}: шкала —', ПАСПОРТ.get(A, ('не у вашому списку',))[0] or '(визначте самостійно)')
print(f'{B}: шкала —', ПАСПОРТ.get(B, ('не у вашому списку',))[0] or '(визначте самостійно)')
print()
print(df[[A, B]].head(6).to_string())

Ваша пара: Стать × помер/жив

Стать: шкала — не у вашому списку
помер/жив: шкала — номінальна

  Стать  помер/жив
0     ч          1
1     ч          2
2     ж          2
3     ж          2
4     ж          2
5     ж          1


In [14]:
ОБРАНА_МІРА = 'χ² Пірсона та коефіцієнт Крамера V'
ЧОМУ_САМЕ_ВОНА = ('Стать і результат лікування (помер/жив) — обидві номінальні: значення можна лише порівнювати '
                  'на рівність, тож придатна міра зв’язку для таблиці спряженості — χ² і V Крамера, а не Пірсон.')

assert ОБРАНА_МІРА and len(ЧОМУ_САМЕ_ВОНА) >= 40, 'Заповніть обидва поля'

from scipy.stats import chi2_contingency, fisher_exact

результат = df['помер/жив'].map({1: 'помер', 2: 'вижив'})
таблиця = pd.crosstab(Ч['Стать'], результат)              # стать вже уніфіковано (ч/ж)
хі2, p, ступені, очікувані = chi2_contingency(таблиця, correction=False)
n = таблиця.values.sum()
V = np.sqrt(хі2 / (n * (min(таблиця.shape) - 1)))

print('Таблиця спряженості (кількість):')
print(таблиця.to_string())
print()
print('Частки в межах статі, %:')
print((100 * таблиця.div(таблиця.sum(axis=1), axis=0)).round(1).to_string())
print()
print(f'χ² = {хі2:.2f}, ступенів свободи = {ступені}, p = {p:.3f}')
print(f'V Крамера = {V:.3f}')
print(f'Найменша очікувана частота = {очікувані.min():.1f} (умова χ²: ≥ 5)')
print(f'Для порівняння: χ² з поправкою Йейтса p = {chi2_contingency(таблиця)[1]:.3f}, '
      f'точний тест Фішера p = {fisher_exact(таблиця)[1]:.3f}')

Таблиця спряженості (кількість):
помер/жив  вижив  помер
Стать                  
ж             32     43
ч             28     18

Частки в межах статі, %:
помер/жив  вижив  помер
Стать                  
ж           42.7   57.3
ч           60.9   39.1

χ² = 3.78, ступенів свободи = 1, p = 0.052
V Крамера = 0.177
Найменша очікувана частота = 22.8 (умова χ²: ≥ 5)
Для порівняння: χ² з поправкою Йейтса p = 0.079, точний тест Фішера p = 0.062


**Інтерпретація.** Серед жінок померли 43 із 75 (57%), серед чоловіків — 18 із 46 (39%).
V Крамера ≈ 0,18 — зв’язок **слабкий**; χ² = 3,78 при p ≈ 0,052 — це **межа**
значущості на рівні 5% (з поправкою Йейтса p ≈ 0,079, у точному тесті Фішера ≈ 0,062).
Отже, в цій вибірці летальність у жінок вища, але надійно відрізнити це від випадковості
не можна. V не має знака: напрям «жінки → частіше помирають» читаємо з таблиці,
а не з коефіцієнта. Причинності з цього не випливає (вік, тяжкість ураження тощо
не контролювалися).

In [15]:
# Контрприклад: Пірсон для двох номінальних змінних
померли = (df['помер/жив'] == 1).astype(int)

# 1) сирий стовпець «Стать» (4 написання) — закодуємо двома «рівноправними» способами
код_порядок_появи = pd.Series(pd.factorize(df['Стать'])[0], index=df.index)
код_за_алфавітом  = df['Стать'].map({'ч': 0, 'чол': 1, 'ж': 2, 'жін': 3})
print('Пірсон, коди статі за порядком появи :', round(код_порядок_появи.corr(померли), 3))
print('Пірсон, коди статі «ч, чол, ж, жін»   :', round(код_за_алфавітом.corr(померли), 3))

# 2) очищена стать (дві категорії) — міняємо місцями нуль та одиницю
r_ч0 = Ч['Стать'].map({'ч': 0, 'ж': 1}).corr(померли)
r_ж0 = Ч['Стать'].map({'ч': 1, 'ж': 0}).corr(померли)
print()
print('Пірсон, ч = 0, ж = 1 :', round(r_ч0, 3))
print('Пірсон, ч = 1, ж = 0 :', round(r_ж0, 3))
print()
print('Те саме спостереження — різні числа й навіть різні знаки (−0,10; 0,07; ±0,18).')
print('Для двох двійкових змінних |r| збігається з V (φ-коефіцієнт), але знак довільний,')
print('а на сирій «брудній» статі число залежить від того, який номер дали якому написанню.')
print('Номінальні мітки не мають ні порядку, ні відстаней — «лінійний зв’язок» між ними не існує.')

Пірсон, коди статі за порядком появи : -0.101
Пірсон, коди статі «ч, чол, ж, жін»   : 0.074

Пірсон, ч = 0, ж = 1 : 0.177
Пірсон, ч = 1, ж = 0 : -0.177

Те саме спостереження — різні числа й навіть різні знаки (−0,10; 0,07; ±0,18).
Для двох двійкових змінних |r| збігається з V (φ-коефіцієнт), але знак довільний,
а на сирій «брудній» статі число залежить від того, який номер дали якому написанню.
Номінальні мітки не мають ні порядку, ні відстаней — «лінійний зв’язок» між ними не існує.


---
# Етап 5. Підготовка змінних до моделі

Шкала визначає не лише статистику, а й спосіб кодування.

| Шкала | Як готують до моделі |
|---|---|
| Номінальна | one-hot кодування; порядок категорій не вводять |
| Порядкова | впорядкована категорія або цілі коди із **заданим** порядком |
| Інтервальна | стандартизація; для дат — перехід до тривалостей |
| Відношень | нормалізація або стандартизація; можливе логарифмування |
| Ідентифікатор | у модель не беруть узагалі |

Для кожного свого стовпця вкажіть спосіб підготовки — і **виконайте його**
щонайменше для трьох стовпців різних шкал.

In [16]:
ПІДГОТОВКА = {
    'ПІП': 'У модель не брати зовсім: ідентифікатор, унікальний для кожного пацієнта, не несе узагальнюваної інформації.',
    'помер/жив': 'Номінальна: бінарне кодування 0/1 (1 = помер) як цільова змінна; коди 1/2 не підставляти як число.',
    'ступінь КТ': 'Порядкова: цілі коди 0–4 із заданим порядком (легкий…критичний); пропуски (65%) позначити окремим індикатором.',
    'Дата розтину': 'Інтервальна: перейти до тривалості в днях від дати поступлення; але ознака відома лише для померлих — витік цілі.',
    'КТ': 'Відношень: розібрати текстові діапазони в частку, далі min-max нормалізація (є межі 0–1); пропуски — індикатор.',
    'Т вище 37,5': 'Відношень: витягти кількість днів із тексту, log1p через праву асиметрію, потім стандартизація (z-оцінка).',
    'Сечов1': 'Відношень: логарифмування (є викиди до 68), потім стандартизація (z-оцінка).',
    'БРпр1': 'Відношень: стандартизація (z-оцінка); значення кратні 1,7 — лабораторне округлення, на тип кодування не впливає.',
}
for c in МОЇ:
    ПІДГОТОВКА.setdefault(c, '')

pd.DataFrame([(c, ПАСПОРТ[c][0], ПІДГОТОВКА[c]) for c in МОЇ],
             columns=['Стовпець', 'Шкала', 'Як готуємо'])

,Стовпець,Шкала,Як готуємо
0,ПІП,ідентифікатор,"У модель не брати зовсім: ідентифікатор, унікальний для ..."
1,помер/жив,номінальна,Номінальна: бінарне кодування 0/1 (1 = помер) як цільова...
2,ступінь КТ,порядкова,Порядкова: цілі коди 0–4 із заданим порядком (легкий…кри...
3,Дата розтину,інтервальна,Інтервальна: перейти до тривалості в днях від дати посту...
4,КТ,відношень,"Відношень: розібрати текстові діапазони в частку, далі m..."
5,"Т вище 37,5",відношень,"Відношень: витягти кількість днів із тексту, log1p через..."
6,Сечов1,відношень,"Відношень: логарифмування (є викиди до 68), потім станда..."
7,БРпр1,відношень,"Відношень: стандартизація (z-оцінка); значення кратні 1,..."


In [17]:
def z(s):
    """стандартизація з перевіркою на сталий стовпець"""
    assert s.nunique() > 1, f'{s.name}: сталий стовпець'
    return (s - s.mean()) / s.std()

def minmax(s):
    assert s.max() > s.min(), f'{s.name}: max = min, ділення на нуль'
    return (s - s.min()) / (s.max() - s.min())

Х = pd.DataFrame(index=df.index)

# номінальна -> 0/1 (one-hot для двох категорій = один стовпець)
Х['помер (1=так)'] = (df['помер/жив'] == 1).astype(int)

# порядкова -> цілі коди із ЗАДАНИМ порядком (NaN лишається NaN)
Х['ступінь КТ (код 0–4)'] = Ч['ступінь КТ'].cat.codes.where(Ч['ступінь КТ'].notna())

# відношень -> лог + стандартизація / min-max / z
Х['Сечов1 (log, z)']      = z(np.log(Ч['Сечов1']))
Х['БРпр1 (z)']            = z(Ч['БРпр1'])
Х['КТ (min-max)']         = minmax(Ч['КТ'])
Х['Т вище 37,5 (log1p, z)'] = z(np.log1p(Ч['Т вище 37,5']))

# інтервальна (дата) -> тривалість у днях від поступлення
тривалість = (Ч['Дата розтину'] - pd.to_datetime(df['Поступлення'])).dt.days
хибні = (тривалість < 0).sum()
print(f'Розтин раніше за поступлення (помилка введення дат): {хибні} запис(ів) -> вважаємо пропуском')
Х['Розтин: днів від поступлення (z)'] = z(тривалість.where(тривалість >= 0).rename('тривалість'))

print()
print(Х.describe().round(2).T[['count', 'mean', 'std', 'min', 'max']].to_string())

Розтин раніше за поступлення (помилка введення дат): 2 запис(ів) -> вважаємо пропуском

                                  count  mean   std   min   max
помер (1=так)                     121.0  0.50  0.50  0.00  1.00
ступінь КТ (код 0–4)               42.0  1.48  1.11  0.00  4.00
Сечов1 (log, z)                   117.0 -0.00  1.00 -1.88  4.58
БРпр1 (z)                         109.0 -0.00  1.00 -1.52  3.43
КТ (min-max)                       43.0  0.50  0.29  0.00  1.00
Т вище 37,5 (log1p, z)             78.0 -0.00  1.00 -2.17  2.84
Розтин: днів від поступлення (z)   59.0 -0.00  1.00 -0.97  4.29


**Перед нормалізацією** обов'язково перевірте варіативність. Стовпець, у якому
всі значення однакові, дасть нуль у знаменнику — і мовчазний `NaN` замість помилки.

In [18]:
for c in МОЇ:
    з = pd.to_numeric(df[c], errors='coerce').dropna()
    if len(з) and з.nunique() == 1:
        print(f'УВАГА: {c} — сталий стовпець, нормалізація дасть ділення на нуль')
    elif len(з) and з.max() == з.min():
        print(f'УВАГА: {c} — max дорівнює min')
print('Перевірку варіативності виконано.')

Перевірку варіативності виконано.


In [19]:
# --- САМОПЕРЕВІРКА 2 ---
assert all(len(ПІДГОТОВКА[c].strip()) >= 15 for c in МОЇ), \
    'Для кожного стовпця вкажіть спосіб підготовки'
assert len(БЕЗГЛУЗДІ) >= 3, 'Етап 3 не завершено'
print('Самоперевірка 2 пройдена.')

Самоперевірка 2 пройдена.


---
# Етап 6. Виправлення чужого аналізу

Відкрийте файл **`ЛР3_помилковий_аналіз.ipynb`**. Це звіт, який виконується
без жодної помилки й містить десять хибних місць.

Для кожного з десяти заповніть рядок нижче:

* **розділ** — номер розділу звіту;
* **що не так** — яка шкала порушена або яке припущення хибне;
* **чому число неправильне** — що саме воно означає насправді;
* **як правильно** — коректний варіант або обґрунтована відмова від показника.

In [20]:
ВИПРАВЛЕННЯ = [
    ('1. Загальна характеристика результату',
     'Середнє для номінальної змінної: коди 1 і 2 — мітки категорій «помер» та «вижив».',
     'Значення 1,50 (точно 1,496) — це лише арифметика над мітками. Воно не означає стану «посередині між смертю й одужанням»; '
     'після перекодування в 0/1 «середнє» стало б 0,50, тобто число залежить від довільного вибору кодів.',
     'Лише частоти й частки: 61 помер (50,4%), 60 вижили (49,6%), мода. Середнє від кодів 0/1 допустиме тільки як частка летальності, а не «середній результат».'),
    ('2. Демографія: розподіл за статтю',
     'Дані непридатні: одні й ті самі категорії записано по-різному (ж / жін, ч / чол); nunique() рахує написання, а не категорії.',
     'Чотири «статі» — артефакт введення даних: «жін» і «чол» — скорочення тих самих «ж» і «ч». Висновок «чотири статі» та потреба пояснювати малі групи хибні.',
     'Спершу уніфікувати ярлики (str.strip().str[0]), потім рахувати частоти: ж — 75 (62%), ч — 46 (38%). Дві категорії, а не чотири.'),
    ('3. Тяжкість ураження легень за даними КТ',
     'Порядкова шкала: сортування за алфавітом замість природного порядку; брудні ярлики (12 написань); середнє кодів.',
     'sort_index() сортує рядки за абеткою, тому «найлегшим» став «важкий», а число 5,9 — це позиція в алфавітному списку з 12 написань, а не тяжкість. '
     'Крім того, відстані між ступенями невідомі, тож середнє кодів і округлення до категорії не мають змісту.',
     'Звести написання до п’яти категорій, задати упорядковану Categorical (легкий < середній < важкий < вкрай важкий < критичний) і рахувати медіану та квартилі (медіана — «середній»); середнє не рахувати.'),
    ('4. Обсяг ураження легень у відсотках',
     'Дані непридатні: стовпець змішує частки (0,3) і текстові діапазони («20-30%», «до 17%»); to_numeric(errors="coerce") мовчки відкидає текст.',
     'З 44 непорожніх записів відкинуто 17, тож 39,6% — це середнє лише за 27 записами (61%), причому відкинуто саме підгрупу з діапазонами; '
     'додатково КТ відсутня у 77 із 121 пацієнтів, тому значення не описує відділення в цілому.',
     'Розібрати діапазони (середина інтервалу), явно вказати кількість записів: за 43 записами середнє ≈ 42,5%, медіана 35%; звітувати про 64% пропусків і не поширювати результат на всіх.'),
    ('5. Сатурація при поступленні',
     'Дані непридатні: запис 0,9 у spO2 (імовірно 90 або частка 0,9) неможливий для живого пацієнта; висновок про «більшість» не випливає із середнього.',
     'Один хибний запис знижує середнє (без нього ≈ 82,7, медіана 84); крім того, середнє нічого не каже про те, у скількох пацієнтів сатурація низька, — для «більшості» потрібна частка.',
     'Перевірити й виправити викид у джерелі, звітувати медіану та квартилі, а для твердження про «більшість» — частку пацієнтів із SpO2 < 90%: 91 зі 120 (≈ 76%).'),
    ('6. Стабільність дат госпіталізації',
     'Коефіцієнт варіації для інтервальної шкали (дати): нуль календаря умовний.',
     'CV = σ / μ залежить від точки відліку: від 1970 р. вийшло 0,6%, від 01.01.2021 — вже 30%. «Однорідність потоку» — артефакт того, що середня дата далеко від умовного нуля.',
     'Для дат — діапазон, медіана, квартилі, σ у днях; однорідність потоку оцінювати за кількістю госпіталізацій за тиждень/місяць. CV рахувати лише для величин із природним нулем (тривалість у днях).'),
    ('7. Зв’язок статі з результатом лікування',
     'Кореляція Пірсона для двох номінальних змінних (стать і результат), закодованих довільними числами; «лінійного зв’язку» між мітками не існує.',
     'r = 0,177 (знак залежить від того, кому дали 0: поміняйте коди — отримаєте −0,177), тож це не «нуль»: χ² = 3,78, p ≈ 0,052; '
     'летальність у жінок 57%, у чоловіків 39%. Висновок «зв’язку не виявлено» хибний.',
     'Таблиця спряженості, χ² і V Крамера = 0,18: зв’язок слабкий, на межі значущості 5%; напрямок читати з таблиці часток.'),
    ('8. Підготовка ознак: нормалізація',
     'Min-max нормалізація застосована до всіх числових стовпців без перевірки: сталі, майже порожні й кодові змінні.',
     'ФЕРИТ2 має лише одне значення (max = min = 1265): у знаменнику нуль, і весь стовпець стає NaN, а не «приведений до [0, 1]». '
     'Нормалізовано і код помер/жив (мітки), а викиди (spO2 = 0,9, сечовина 68) стискають решту значень у вузький піддіапазон.',
     'Перед нормалізацією перевіряти nunique() > 1 і частку непорожніх; не нормалізувати ідентифікатори й коди; для викидів — логарифм або робастне масштабування.'),
    ('9. Очищення набору від пропусків',
     'Дані непридатні: df.dropna() на широкому наборі, де багато стовпців майже порожні, а пропуск часто означає «не вимірювалося».',
     'dropna() залишає 0 рядків зі 121 (кожен пацієнт має пропуск, ФЕРИТ2 заповнений лише в одного), тож «повністю заповнений набір» порожній, '
     'а висновок про придатність для моделі хибний.',
     'Спершу відкинути стовпці з надмірною часткою пропусків (скажімо, > 30%), потім оцінити рядки: лишається ≈ 52 рядки з 58 стовпцями (втрата 57% пацієнтів) — або імпутація чи модель, що працює з пропусками.'),
    ('10. Рейтинг важливості ознак',
     'Витік цілі: Дата розтину існує лише в померлих; NaT перетворено на число −9,2·10^18; кореляція Пірсона рахується на різній кількості спостережень.',
     'r = 1,00 виникає, бо «дата розтину заповнена» ⇔ «пацієнт помер»: ознака містить саму відповідь, а не прогноз. '
     'Решта рангу — кореляції на різних підвибірках (ІЛ6-2 — лише 6 пацієнтів), тому порядок ненадійний.',
     'Виключити ознаки, відомі лише після результату (дата розтину, виписка, ускладнення); NaT не кодувати числами; ранжувати на ознаках із достатньою заповненістю й вказувати n.'),
]

assert len(ВИПРАВЛЕННЯ) == 10, f'Знайдено {len(ВИПРАВЛЕННЯ)} помилок із 10'
for р in ВИПРАВЛЕННЯ:
    assert len(р) == 4 and all(str(x).strip() for x in р), 'Заповніть усі чотири поля'
    assert len(str(р[2])) >= 30 and len(str(р[3])) >= 25, 'Пояснення закороткі'

pd.DataFrame(ВИПРАВЛЕННЯ, columns=['Розділ', 'Що не так', 'Чому число неправильне', 'Як правильно'])

,Розділ,Що не так,Чому число неправильне,Як правильно
0,1. Загальна характеристика результату,Середнє для номінальної змінної: коди 1 і 2 — мітки кате...,"Значення 1,50 (точно 1,496) — це лише арифметика над міт...","Лише частоти й частки: 61 помер (50,4%), 60 вижили (49,6..."
1,2. Демографія: розподіл за статтю,Дані непридатні: одні й ті самі категорії записано по-рі...,Чотири «статі» — артефакт введення даних: «жін» і «чол» ...,"Спершу уніфікувати ярлики (str.strip().str[0]), потім ра..."
2,3. Тяжкість ураження легень за даними КТ,Порядкова шкала: сортування за алфавітом замість природн...,"sort_index() сортує рядки за абеткою, тому «найлегшим» с...","Звести написання до п’яти категорій, задати упорядковану..."
3,4. Обсяг ураження легень у відсотках,"Дані непридатні: стовпець змішує частки (0,3) і текстові...","З 44 непорожніх записів відкинуто 17, тож 39,6% — це сер...","Розібрати діапазони (середина інтервалу), явно вказати к..."
4,5. Сатурація при поступленні,"Дані непридатні: запис 0,9 у spO2 (імовірно 90 або частк...","Один хибний запис знижує середнє (без нього ≈ 82,7, меді...","Перевірити й виправити викид у джерелі, звітувати медіан..."
5,6. Стабільність дат госпіталізації,Коефіцієнт варіації для інтервальної шкали (дати): нуль ...,CV = σ / μ залежить від точки відліку: від 1970 р. вийшл...,"Для дат — діапазон, медіана, квартилі, σ у днях; однорід..."
6,7. Зв’язок статі з результатом лікування,Кореляція Пірсона для двох номінальних змінних (стать і ...,"r = 0,177 (знак залежить від того, кому дали 0: поміняйт...","Таблиця спряженості, χ² і V Крамера = 0,18: зв’язок слаб..."
7,8. Підготовка ознак: нормалізація,Min-max нормалізація застосована до всіх числових стовпц...,ФЕРИТ2 має лише одне значення (max = min = 1265): у знам...,Перед нормалізацією перевіряти nunique() > 1 і частку не...
8,9. Очищення набору від пропусків,"Дані непридатні: df.dropna() на широкому наборі, де бага...",dropna() залишає 0 рядків зі 121 (кожен пацієнт має проп...,Спершу відкинути стовпці з надмірною часткою пропусків (...
9,10. Рейтинг важливості ознак,Витік цілі: Дата розтину існує лише в померлих; NaT пере...,"r = 1,00 виникає, бо «дата розтину заповнена» ⇔ «пацієнт...","Виключити ознаки, відомі лише після результату (дата роз..."


### Виправте щонайменше три помилки в коді

Оберіть три з десяти та покажіть правильне обчислення поруч із неправильним.

In [21]:
# ---------- Розділ 3: ступінь КТ ----------
print('=== Розділ 3 ===')
хибно = df['ступінь КТ'].value_counts().sort_index()
print('НЕПРАВИЛЬНО: «найлегший» за алфавітом —', repr(хибно.index[0]), '; усього написань:', len(хибно))
print('ПРАВИЛЬНО: упорядковані категорії')
print(Ч['ступінь КТ'].value_counts(sort=False).to_string())
ст = допустима_статистика('ступінь КТ', 'порядкова')
print('Медіана:', ст['медіана'], '| Q1:', ст['Q1'], '| Q3:', ст['Q3'])

# ---------- Розділ 5: сатурація ----------
print()
print('=== Розділ 5 ===')
s = df['spO2 без дот'].dropna()
print('НЕПРАВИЛЬНО: середнє', round(s.mean(), 2), '(вплив запису', s.min(), ')')
print('ПРАВИЛЬНО: медіана', s.median(), '| Q1–Q3', s.quantile(.25), '–', s.quantile(.75),
      '| середнє без неможливих значень (<30):', round(s[s >= 30].mean(), 2))
print(f'Частка пацієнтів із SpO2 < 90%: {(s < 90).sum()} із {len(s)} = {100 * (s < 90).mean():.0f}%')

# ---------- Розділ 7: стать і результат ----------
print()
print('=== Розділ 7 ===')
print('НЕПРАВИЛЬНО: Пірсон між кодами', round(Ч['Стать'].map({'ч': 0, 'ж': 1}).corr((df['помер/жив'] == 1).astype(int)), 3),
      '(знак залежить від кодування)')
print(f'ПРАВИЛЬНО: χ² = {хі2:.2f}, p = {p:.3f}, V = {V:.3f}')
print('Летальність жінок / чоловіків, %:',
      round(100 * таблиця.loc['ж', 'помер'] / таблиця.loc['ж'].sum(), 1), '/',
      round(100 * таблиця.loc['ч', 'помер'] / таблиця.loc['ч'].sum(), 1))

# ---------- Розділ 8: нормалізація ----------
print()
print('=== Розділ 8 ===')
числові = df.select_dtypes('number')
різниця = числові.max() - числові.min()
print('НЕПРАВИЛЬНО: у звіті ділення на нуль у стовпцях:', list(різниця[різниця == 0].index))
зламані = числові[різниця[різниця == 0].index]
print('Числових значень у них після звітної формули:', ((зламані - зламані.min()) / (зламані.max() - зламані.min())).notna().sum().sum(),
      '(було непорожніх:', зламані.notna().sum().sum(), ')')
придатні = [c for c in числові.columns
            if числові[c].nunique() > 1 and числові[c].notna().mean() >= 0.5 and c != 'помер/жив']
норм = (числові[придатні] - числові[придатні].min()) / (числові[придатні].max() - числові[придатні].min())
print(f'ПРАВИЛЬНО: нормалізовано {len(придатні)} із {числові.shape[1]} числових стовпців; '
      f'мінімум {норм.min().min():.0f}, максимум {норм.max().max():.0f}')

# ---------- Розділ 9: dropna ----------
print()
print('=== Розділ 9 ===')
print('НЕПРАВИЛЬНО: після df.dropna() рядків', len(df.dropna()), 'із', len(df))
лишити = df.drop(columns='Примітка').dropna(axis=1, thresh=int(0.7 * len(df)))
print(f'ПРАВИЛЬНО: після відкидання стовпців із > 30% пропусків лишилось {лишити.shape[1]} стовпців, '
      f'повних рядків: {len(лишити.dropna())} із {len(df)}')

=== Розділ 3 ===
НЕПРАВИЛЬНО: «найлегший» за алфавітом — 'важкий' ; усього написань: 12
ПРАВИЛЬНО: упорядковані категорії
ступінь КТ
легкий           7
середній        18
важкий          10
вкрай важкий     4
критичний        3
Медіана: середній | Q1: середній | Q3: важкий

=== Розділ 5 ===
НЕПРАВИЛЬНО: середнє 82.05 (вплив запису 0.9 )
ПРАВИЛЬНО: медіана 84.0 | Q1–Q3 78.75 – 89.0 | середнє без неможливих значень (<30): 82.73
Частка пацієнтів із SpO2 < 90%: 91 із 120 = 76%

=== Розділ 7 ===
НЕПРАВИЛЬНО: Пірсон між кодами 0.177 (знак залежить від кодування)
ПРАВИЛЬНО: χ² = 3.78, p = 0.052, V = 0.177
Летальність жінок / чоловіків, %: 57.3 / 39.1

=== Розділ 8 ===
НЕПРАВИЛЬНО: у звіті ділення на нуль у стовпцях: ['ФЕРИТ2']
Числових значень у них після звітної формули: 0 (було непорожніх: 1 )
ПРАВИЛЬНО: нормалізовано 34 із 40 числових стовпців; мінімум 0, максимум 1

=== Розділ 9 ===
НЕПРАВИЛЬНО: після df.dropna() рядків 0 із 121
ПРАВИЛЬНО: після відкидання стовпців із > 30% пропусків лиши

---
# Етап 7. Висновки та індивідуальне питання

In [22]:
ВИСНОВКИ = [
    'З восьми моїх стовпців лише чотири (КТ, Т вище 37,5, Сечов1, БРпр1) мають шкалу відношень, і два з них, КТ та Т вище 37,5, '
    'записано вперемішку з текстом, тож статистику можна рахувати тільки після очищення: у КТ 43 числа з 44 записів, у температурі 78.',
    'Результат лікування (помер/жив) — номінальна змінна, тому його «середнє» 1,50 безглузде; зв’язок зі статтю оцінюється лише через χ² = 3,78 '
    'і V = 0,18: слабкий, на межі значущості (p ≈ 0,052), летальність у жінок 57%, у чоловіків 39%, але лише після уніфікації чотирьох написань статі до двох.',
    'Дата розтину (інтервальна шкала) заповнена лише у 61 померлого, тому для прогнозу летальності вона не придатна (витік цілі), '
    'а коефіцієнт варіації дат безглуздий; ПІП — ідентифікатор, у модель і в статистику не береться.',
]
assert len(ВИСНОВКИ) >= 3 and all(len(в) >= 60 for в in ВИСНОВКИ), \
    'Потрібні три змістовні висновки'
for i, в in enumerate(ВИСНОВКИ, 1):
    print(f'{i}. {в}')

1. З восьми моїх стовпців лише чотири (КТ, Т вище 37,5, Сечов1, БРпр1) мають шкалу відношень, і два з них, КТ та Т вище 37,5, записано вперемішку з текстом, тож статистику можна рахувати тільки після очищення: у КТ 43 числа з 44 записів, у температурі 78.
2. Результат лікування (помер/жив) — номінальна змінна, тому його «середнє» 1,50 безглузде; зв’язок зі статтю оцінюється лише через χ² = 3,78 і V = 0,18: слабкий, на межі значущості (p ≈ 0,052), летальність у жінок 57%, у чоловіків 39%, але лише після уніфікації чотирьох написань статі до двох.
3. Дата розтину (інтервальна шкала) заповнена лише у 61 померлого, тому для прогнозу летальності вона не придатна (витік цілі), а коефіцієнт варіації дат безглуздий; ПІП — ідентифікатор, у модель і в статистику не береться.


In [23]:
print('ВАШЕ ПИТАННЯ:'); print(В['питання']); print()

# де формула (x − min) / (max − min) ламається: стовпець, у якому max = min
числові = df.select_dtypes('number')
різниця = числові.max() - числові.min()
ламаються = list(різниця[різниця == 0].index)
print('Числові стовпці з max = min:', ламаються)
print('Непорожніх значень у них:', числові[ламаються].notna().sum().to_dict())
результат_формули = (числові[ламаються] - числові[ламаються].min()) / різниця[ламаються]
print('Значень у [0, 1] після формули:', результат_формули.notna().sum().sum(), '— усе перетворилось на NaN')
print()

ВІДПОВІДЬ = ('Формула зламається на стовпці «ФЕРИТ2»: у ньому єдине непорожнє значення (1265), тому max = min і знаменник max − min дорівнює нулю. '
             'Чисельник x − min теж нуль, отже виходить 0/0 = NaN, і замість діапазону [0, 1] весь стовпець перетворюється на пропуски — '
             'pandas при цьому не видає помилки. Стовпець сталий і нічого не розрізняє, тож для моделі він марний. '
             'Перед нормалізацією треба перевіряти nunique() > 1 і max > min, а такі стовпці (а також майже порожні на кшталт ФЕРИТ1 з двома значеннями '
             'чи ІЛ6-2 із шістьма) виключати або не нормалізувати.')
assert len(ВІДПОВІДЬ.strip()) >= 200, 'Відповідь має бути розгорнутою (від 200 символів)'
print(ВІДПОВІДЬ)

ВАШЕ ПИТАННЯ:
Ви хочете нормалізувати всі числові стовпці формулою (x − min) / (max − min). Знайдіть у файлі стовпець, на якому ця формула зламається, і поясніть чому.

Числові стовпці з max = min: ['ФЕРИТ2']
Непорожніх значень у них: {'ФЕРИТ2': 1}
Значень у [0, 1] після формули: 0 — усе перетворилось на NaN

Формула зламається на стовпці «ФЕРИТ2»: у ньому єдине непорожнє значення (1265), тому max = min і знаменник max − min дорівнює нулю. Чисельник x − min теж нуль, отже виходить 0/0 = NaN, і замість діапазону [0, 1] весь стовпець перетворюється на пропуски — pandas при цьому не видає помилки. Стовпець сталий і нічого не розрізняє, тож для моделі він марний. Перед нормалізацією треба перевіряти nunique() > 1 і max > min, а такі стовпці (а також майже порожні на кшталт ФЕРИТ1 з двома значеннями чи ІЛ6-2 із шістьма) виключати або не нормалізувати.


---
# Етап 8. Декларація про використання інструментів ШІ

Користуватися мовними моделями дозволено. Це нормальна інженерна практика.

**Обов'язковою є прозорість:** ви вказуєте, де саме і як скористалися інструментом —
так само, як у науковій статті вказують використане програмне забезпечення.
Задекларована допомога порушенням не є.

Порушенням є інше: здати роботу, яку ви не можете пояснити. На захисті вас
попросять показати конкретний рядок свого коду, пояснити, чому він саме такий,
і внести в нього невелику зміну на місці.

Для кожного етапу оберіть одне з формулювань:

* `'ні'` — інструментами ШІ не користувався
* `'синтаксис'` — питав про назву функції або синтаксис, код писав сам
* `'код, перевірено'` — код згенеровано, я його прочитав, перевірив і розумію
* `'текст, перероблено'` — згенеровано чернетку тексту, яку я переписав по суті
* `'текст, як є'` — згенерований текст залишено майже без змін

In [24]:
ДЕКЛАРАЦІЯ = {
    'етап 1 — паспорт змінних':        'ні',
    'етап 2 — допустимі статистики':   'синтаксис',
    'етап 3 — безглузді числа':        'синтаксис',
    'етап 4 — міра зв’язку':           'синтаксис',
    'етап 5 — підготовка до моделі':   'синтаксис',
    'етап 6 — виправлення аналізу':    'ні',
    'етап 7 — висновки й відповідь':   'ні',
}

ЩО_САМЕ_Я_ПЕРЕВІРИЛА = ' Визначила шкали вимірювання восьми стовпців свого варіанта (ідентифікатор, номінальна, порядкова, інтервальна, відношень) і обґрунтувала кожну. Очистила «брудні» дані: звела 12 написань ступеня КТ до п’яти, уніфікувала стать до двох категорій, розібрала текстові діапазони в КТ і текст у «Т вище 37,5». Для кожного стовпця порахувала лише допустимі для його шкали статистики та навела п’ять безглуздих чисел. Для пари «стать × помер/жив» обрала χ² і V Крамера (V = 0,18, p ≈ 0,052) і показала, що кореляція Пірсона дає різні числа залежно від кодування. Підготувала змінні до моделі відповідно до шкал (кодування, логарифм, стандартизація, тривалості замість дат). Знайшла і виправила десять помилок у чужому звіті. Знайшла, що нормалізація min-max ламається на стовпці ФЕРИТ2, бо там одне значення і max = min. Перезапустила ноутбук повністю, помилок немає.)'
ПІДТВЕРДЖУЮ =True     # я можу пояснити кожен рядок свого коду
                        # і внести в нього зміни на вимогу під час захисту

In [26]:
# --- ФІНАЛЬНА САМОПЕРЕВІРКА ---
ФОРМУЛЮВАННЯ = {'ні', 'синтаксис', 'код, перевірено', 'текст, перероблено', 'текст, як є'}

перевірки = {
    'заповнено титульні поля':        bool(ПРІЗВИЩЕ_ІМЯ and ГРУПА and НОМЕР),
    'етап 1: паспорт із 8 стовпців':  all(ПАСПОРТ[c][0] and ПАСПОРТ[c][1] for c in МОЇ),
    'етап 3: три безглузді числа':    len(БЕЗГЛУЗДІ) >= 3,
    'етап 4: міру обрано й пояснено': bool(ОБРАНА_МІРА and len(ЧОМУ_САМЕ_ВОНА) >= 40),
    'етап 5: підготовку описано':     all(len(ПІДГОТОВКА[c].strip()) >= 15 for c in МОЇ),
    'етап 6: знайдено 10 помилок':    len(ВИПРАВЛЕННЯ) == 10,
    'етап 7: три висновки':           len(ВИСНОВКИ) >= 3,
    'етап 7: відповідь на питання':   len(ВІДПОВІДЬ.strip()) >= 200,
    'декларацію заповнено':           all(v in ФОРМУЛЮВАННЯ for v in ДЕКЛАРАЦІЯ.values()),
    'опис перевірки ≥ 120 символів':  len(ЩО_САМЕ_Я_ПЕРЕВІРИЛА.strip()) >= 120,
    'підтвердження авторства':        ПІДТВЕРДЖУЮ is True,
}

print('ЧЕК-ЛИСТ ГОТОВНОСТІ')
print('=' * 52)
for назва, ок in перевірки.items():
    print(('  ГОТОВО     ' if ок else '  НЕ ГОТОВО ') + назва)
print('=' * 52)

assert all(перевірки.values()), 'Робота не готова до здачі — див. чек-лист вище'
print()
print(f'Робота варіанта {В["варіант"]} готова до захисту.')
print(f'{ПРІЗВИЩЕ_ІМЯ}, {ГРУПА}')

ЧЕК-ЛИСТ ГОТОВНОСТІ
  ГОТОВО     заповнено титульні поля
  ГОТОВО     етап 1: паспорт із 8 стовпців
  ГОТОВО     етап 3: три безглузді числа
  ГОТОВО     етап 4: міру обрано й пояснено
  ГОТОВО     етап 5: підготовку описано
  ГОТОВО     етап 6: знайдено 10 помилок
  ГОТОВО     етап 7: три висновки
  ГОТОВО     етап 7: відповідь на питання
  ГОТОВО     декларацію заповнено
  ГОТОВО     опис перевірки ≥ 120 символів
  ГОТОВО     підтвердження авторства

Робота варіанта 7 готова до захисту.
Ільящук Марта, ШІ-31


---
### Перед здачею

Виконайте **Kernel → Restart & Run All**. Ноутбук має пройти згори донизу
без жодної помилки — це перше, що перевіряється на захисті.